# Signal research (WORKFLOW Stage 1–2)

Characterise a raw signal *before* trusting a backtest: IC, decay / half-life, turnover, by-regime.

> **Parity rule:** this notebook is an *exploration front-end* — it only imports from `mt5_trader.*` and never reimplements signal/engine logic. The same `signal()` runs here, in the backtester, and live. Keep strategy logic causal (no `.shift(-1)`); `forward_return` below is measurement-only.

Run **Kernel → Restart & Run All** before trusting a result. Runs on real ticks (`data/processed`) if present, else deterministic synthetic bars.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from mt5_trader.pipeline.data import load_or_synthetic, bars_per_year
from mt5_trader.pipeline import diagnostics
from mt5_trader.strategies import REGISTRY

SYMBOL, EVERY = "XAUUSD", "5m"

## 1. Load bars

In [ ]:
bars, is_real = load_or_synthetic(SYMBOL, EVERY)
print(("REAL" if is_real else "SYNTHETIC"), "bars:", bars.shape)
bars.head()

## 2. Pick a signal
`name` is any key in `REGISTRY` — swap freely.

In [ ]:
name = "vwap_trend"
sig = REGISTRY[name]().signal(bars)

## 3. Diagnostics
IC vs forward return, rank-IC, turnover, regime slice.

In [ ]:
rep = diagnostics.diagnose(bars, sig)
print(f"IC {rep['ic']:+.4f}  rankIC {rep['rank_ic']:+.4f}  "
      f"turnover {rep['turnover']:.0f}  active {rep['active_frac']:.2f}")
print("half-life:", rep['decay']['half_life'], "bars")
rep['by_regime']

## 4. Signal decay
Where does the edge live across horizons? Sets the natural holding period.

In [ ]:
ics = rep['decay']['ic_by_horizon']
plt.figure(figsize=(7, 3))
plt.plot(list(ics), list(ics.values()), marker='o')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('horizon (bars)'); plt.ylabel('IC')
plt.title(f'{name} signal decay'); plt.show()

## Graduate
If an edge holds week-by-week with a sane holding period, promote it: register a strategy, then gate it with `uv run scripts/strategy.py validate <name>`.